In [1]:
import sys
import os
import joblib
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import load_model
import time
import json
import tracemalloc

# IPIE / PySCF Imports
from pyscf import gto, scf, lib
from ipie.utils.from_pyscf import gen_ipie_input_from_pyscf_chk
from ipie.qmc.afqmc import AFQMC
from ipie.utils.mpi import MPIHandler
import ipie.estimators.local_energy_sd

# IPIE Analysis Imports
from ipie.analysis.autocorr import reblock_by_autocorr
from ipie.analysis.extraction import extract_observable

# Check for GPU (CuPy) support
try:
    import cupy as cp
    has_cupy = True
except ImportError:
    cp = None
    has_cupy = False

# =============================================================================
# 0. CONFIGURATION & ML LOADING
# =============================================================================
N_ATOMS = 10
TEST_SEED = 999 
SYSTEM_NAME = f"H{N_ATOMS}"
WALKERS = 2048
NUM_BLOCKS = 50
DEPLOY_DIR = "deployment_objects"
MODEL_PATH = os.path.join(DEPLOY_DIR, f"NN_{SYSTEM_NAME}_DeltaHF.keras")

@tf.keras.utils.register_keras_serializable()
def log_cosh_loss(y_true, y_pred):
    y_true = tf.cast(tf.reshape(y_true, tf.shape(y_pred)), dtype=y_pred.dtype)
    return tf.reduce_mean(tf.math.log(tf.math.cosh(y_pred - y_true)))

# =============================================================================
# 1. HELPER FUNCTIONS
# =============================================================================
def get_dynamic_operators(mol):
    S = mol.intor('int1e_ovlp')
    h_core_ao = mol.intor('int1e_nuc') + mol.intor('int1e_kin')
    e, v = np.linalg.eigh(S)
    mask = e > 1e-15
    S_inv_sqrt = v[:, mask] @ np.diag(e[mask]**(-0.5)) @ v[:, mask].T
    S_sqrt = v[:, mask] @ np.diag(e[mask]**(0.5)) @ v[:, mask].T
    h_core_lowdin = S_inv_sqrt.T @ h_core_ao @ S_inv_sqrt
    return S_inv_sqrt, S_sqrt, h_core_lowdin, S

# =============================================================================
# 2. THE ML PROXY (Device-Aware Version)
# =============================================================================
def create_ml_local_energy_patch(
    ml_model, X_scaler, y_scaler, 
    P_hf_ref, E_hf_ref, S_sqrt, h_core_dyn, C_a, 
    use_gpu, n_atoms
):
    P_hf_ref_np = np.asarray(P_hf_ref)
    S_sqrt_np = np.asarray(S_sqrt)
    h_core_dyn_np = np.asarray(h_core_dyn)
    C_a_np = np.asarray(C_a)

    @tf.function(reduce_retracing=True)
    def fast_predict(inputs):
        return ml_model(inputs, training=False)

    call_counter = {"count": 0}

    def local_energy_single_det_uhf(system, hamiltonian, walkers, trial):
        if call_counter["count"] < 1:
            print(f"\n[ML-PROXY] Intercepted call. CPU Algebra -> Device-Aware Casting.")
            call_counter["count"] += 1

        nwalkers = walkers.nwalkers
        nalpha = trial.nalpha
        
        def to_cpu(obj):
            return obj.get() if hasattr(obj, 'get') else obj

        phi_a = to_cpu(walkers.phia) if hasattr(walkers, 'phia') else to_cpu(walkers.phi[:, :, :nalpha])
        phi_b = to_cpu(walkers.phib) if hasattr(walkers, 'phib') else to_cpu(walkers.phi[:, :, nalpha:])
        Psi_T_a = to_cpu(trial.psi[:, :nalpha])
        Psi_T_b = to_cpu(trial.psi[:, nalpha:])

        O_a = np.einsum('ui, wuj -> wij', Psi_T_a.conj(), phi_a)
        O_b = np.einsum('ui, wuj -> wij', Psi_T_b.conj(), phi_b)
        invO_a = np.linalg.inv(O_a)
        invO_b = np.linalg.inv(O_b)
        
        G_mo_a = np.einsum('wvi, wiu -> wvu', phi_a, np.einsum('wij, ju -> wiu', invO_a, Psi_T_a.conj().T))
        G_mo_b = np.einsum('wvi, wiu -> wvu', phi_b, np.einsum('wij, ju -> wiu', invO_b, Psi_T_b.conj().T))
        
        P_ao = (np.einsum("qi, wij, pj -> wqp", C_a_np, G_mo_a, C_a_np.conj()) + 
                np.einsum("qi, wij, pj -> wqp", C_a_np, G_mo_b, C_a_np.conj()))

        P_lowdin = np.einsum('ai, wib, bj -> waj', S_sqrt_np, P_ao, S_sqrt_np)
        delta_P = P_lowdin - P_hf_ref_np
        E_1B_delta = np.einsum('ij, wji -> w', h_core_dyn_np, delta_P).real

        X_input = np.stack([np.real(delta_P), np.imag(delta_P)], axis=-1).astype(np.float32)
        X_flat = X_input.reshape(nwalkers, -1)
        X_scaled = X_scaler.transform(X_flat).reshape(nwalkers, n_atoms, n_atoms, 2)

        preds_scaled = fast_predict(X_scaled).numpy()
        E_corr_delta = y_scaler.inverse_transform(preds_scaled).flatten()

        energy_out = np.zeros((nwalkers, 3), dtype=np.complex128)
        energy_out[:, 0] = E_hf_ref + E_1B_delta + E_corr_delta
        energy_out[:, 1] = E_hf_ref + E_1B_delta
        energy_out[:, 2] = E_corr_delta
        
        is_gpu_walker = hasattr(walkers.weight, 'device') or 'cupy' in str(type(walkers.weight))
        if is_gpu_walker and cp is not None:
            return cp.asarray(energy_out)
        return energy_out

    return local_energy_single_det_uhf

# =============================================================================
# 3. MAIN INITIALIZATION & PATCHING
# =============================================================================
comm = MPIHandler()
rank = comm.rank

if rank == 0:
    print(f">>> LOADING ML ASSETS...")

try:
    ml_model = load_model(MODEL_PATH, custom_objects={"log_cosh_loss": log_cosh_loss})
    X_scaler = joblib.load(os.path.join(DEPLOY_DIR, "X_scaler.save"))
    y_scaler = joblib.load(os.path.join(DEPLOY_DIR, "y_scaler.save"))
    P_hf_ref = np.load(os.path.join(DEPLOY_DIR, "P_hf_lowdin.npy"))
except Exception as e:
    if rank == 0:
        print(f"Error loading ML assets: {e}")
    sys.exit(1)

if rank == 0:
    mol = gto.M(atom=[("H", 0.74 * j, 0, 0) for j in range(N_ATOMS)], basis="sto-6g", verbose=0)
    mf = scf.UHF(mol)
    mf.kernel()
    gen_ipie_input_from_pyscf_chk(mf.chkfile, hamil_file=f"ham_h{N_ATOMS}.h5", wfn_file=f"wfn_h{N_ATOMS}.h5", verbose=0)
    
    E_HF = mf.e_tot
    C_a = mf.mo_coeff[0] if np.ndim(mf.mo_coeff) == 3 else mf.mo_coeff
    _, S_sqrt, h_core, _ = get_dynamic_operators(mol)
else:
    E_HF = C_a = S_sqrt = h_core = None

E_HF = comm.comm.bcast(E_HF, root=0)
C_a = comm.comm.bcast(C_a, root=0)
S_sqrt = comm.comm.bcast(S_sqrt, root=0)
h_core = comm.comm.bcast(h_core, root=0)

afqmc = AFQMC.build_from_hdf5(
    num_elec=(N_ATOMS//2, N_ATOMS//2), ham_file=f"ham_h{N_ATOMS}.h5",
    wfn_file=f"wfn_h{N_ATOMS}.h5", num_walkers=WALKERS, num_blocks=NUM_BLOCKS,    
    num_steps_per_block=1, verbose=0, seed=TEST_SEED 
)
afqmc.mpi_handler = comm

ml_proxy = create_ml_local_energy_patch(
    ml_model, X_scaler, y_scaler, P_hf_ref, E_HF, S_sqrt, h_core, C_a, has_cupy, N_ATOMS
)

original_func = ipie.estimators.local_energy_sd.local_energy_single_det_uhf
for mod_name, module in list(sys.modules.items()):
    if module is None or not mod_name.startswith("ipie"): continue
    try:
        for attr_name, attr_value in vars(module).items():
            if attr_value is original_func:
                setattr(module, attr_name, ml_proxy)
    except: pass

if hasattr(afqmc, 'propagator'):
    afqmc.propagator.local_energy = ml_proxy

# =============================================================================
# 4. RUN WITH PROFILING
# =============================================================================
if rank == 0: 
    print("\n" + "#"*60 + "\n### STARTING ML-AFQMC PRODUCTION RUN ###\n" + "#"*60)
    tracemalloc.start()
    start_time = time.perf_counter()

# Run the ML-patched engine
afqmc.run()

if rank == 0:
    end_time = time.perf_counter()
    current_mem, peak_mem = tracemalloc.get_traced_memory()
    tracemalloc.stop()

# =============================================================================
# 5. DATA EXTRACTION, ANALYSIS & SAVING
# =============================================================================
if rank == 0:
    print("\n" + "="*50)
    print(">>> PERFORMING AUTOCORRELATION ANALYSIS")
    print("="*50)
    
    final_energy = None
    final_error = None
    
    try:
        # Extract the observable data from the estimators file
        estimator_filename = afqmc.estimators.filename
        qmc_data = extract_observable(estimator_filename, "energy")
        
        # Discard the first block to account for equilibration
        y_energy = qmc_data["ETotal"][1:]
        
        # Perform autocorrelation reblocking to get true statistical error
        df_ac = reblock_by_autocorr(y_energy, verbose=0)
        
        # Extract the relevant values from the dataframe
        final_energy = float(df_ac["ETotal_ac"].iloc[0])
        final_error = float(df_ac["ETotal_error_ac"].iloc[0])
        
        print(f"  Final Reblocked Energy: {final_energy:.6f} Ha")
        print(f"  Final Reblocked Error:  {final_error:.6f} Ha")
        
    except Exception as e:
        print(f"  [Warning] Autocorrelation analysis failed: {e}")
        print("  Falling back to raw standard error of the mean.")
        if 'y_energy' in locals() and len(y_energy) > 0:
            final_energy = float(np.mean(y_energy))
            final_error = float(np.std(y_energy) / np.sqrt(len(y_energy)))
            print(f"  Fallback Raw Mean Energy: {final_energy:.6f} ± {final_error:.6f} Ha")
    
    n_basis = N_ATOMS  
    total_time = end_time - start_time
    time_per_block = total_time / NUM_BLOCKS
    peak_mem_mb = peak_mem / (1024 ** 2)
    
    flops_fe_per_step = 8 * WALKERS * (n_basis ** 3) 
    total_flops = flops_fe_per_step * NUM_BLOCKS
    teraflops_per_sec = (total_flops / total_time) / 1e12

    metrics = {
        "system": SYSTEM_NAME,
        "n_atoms": N_ATOMS,
        "n_basis": n_basis,
        "total_walkers": WALKERS,
        "num_blocks": NUM_BLOCKS,
        "results": {
            "final_energy_ha": final_energy,
            "final_error_ha": final_error
        },
        "timing": {
            "total_wall_time_sec": round(total_time, 4),
            "time_per_block_sec": round(time_per_block, 4)
        },
        "memory": {
            "peak_rss_mb": round(peak_mem_mb, 2)
        },
        "compute": {
            "est_flops_per_step_physics": flops_fe_per_step,
            "est_total_flops_physics": total_flops,
            "est_tflops_throughput_physics": round(teraflops_per_sec, 6),
            "nn_flops": "Dependent on loaded Keras model architecture"
        }
    }

    print("\n" + "="*50)
    print(">>> ML PROXY SCALING METRICS SUMMARY")
    print("="*50)
    print(f"  Final Energy:         {final_energy} ± {final_error} Ha")
    print(f"  Total Wall Time:      {total_time:.4f} sec")
    print(f"  Time Per Block:       {time_per_block:.4f} sec")
    print(f"  Peak Memory (Rank 0): {peak_mem_mb:.2f} MB")
    print("="*50)

    # Save to disk
    save_path = f"ml_proxy_metrics_{SYSTEM_NAME}.json"
    with open(save_path, "w") as f:
        json.dump(metrics, f, indent=4)
    print(f"Metrics saved to {save_path}")

2026-02-27 13:50:30.395315: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


>>> LOADING ML ASSETS...


I0000 00:00:1772218238.345995 1654129 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 766 MB memory:  -> device: 0, name: NVIDIA A40, pci bus id: 0000:23:00.0, compute capability: 8.6


# random seed is 999

############################################################
### STARTING ML-AFQMC PRODUCTION RUN ###
############################################################
            Block                   Weight            WeightFactor            HybridEnergy                  ENumer                  EDenom                  ETotal                  E1Body                  E2Body

[ML-PROXY] Intercepted call. CPU Algebra -> Device-Aware Casting.
                0   2.0480000000000000e+03  2.0480000000000000e+03  0.0000000000000000e+00 -1.0441186452672062e+04  2.0480000000000000e+03 -5.0982355725937802e+00 -5.0965095726060961e+00 -1.7259999876841903e-03
                1   2.0616780949217637e+03  2.0480000000000000e+03 -2.6630942934985069e+00 -1.0512960377430787e+04  2.0616780949217637e+03 -5.0992249485144443e+00 -5.0965077286629761e+00 -2.7172198514668227e-03
                2   2.0617445384773637e+03  2.0480000000000000e+03 -2.6759482301032143e+00 -1.0515274049344371e+04 

In [3]:
(-5.118546327547921 - -5.118739756834181)*630

0.12186045034392912

In [4]:
0.12186045034392912/10

0.012186045034392912

In [5]:
0.012186045034392912*110

1.3404649537832203